## 01. 라이브러리 불러오기

In [2]:
# ==================================================
# 1. 라이브러리 불러오기
# ==================================================

import os

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from PIL import Image

## 02. 기본 설정

In [3]:
# ==================================================
# 2. 기본 설정
# ==================================================

# 데이터셋 위치
dataset_path = "../dataset"

# 모델 저장 위치
model_path = "models"

# 이미지 크기
image_size = 128

# 한 번에 학습할 이미지 수
batch_size = 4

# 학습 횟수
epochs = 20

# 학습률
learning_rate = 0.001


# 모델 저장 폴더 만들기
os.makedirs(
    model_path,
    exist_ok=True
)

## 03. 이미지 전처리

In [4]:
# ==================================================
# 3. 이미지 전처리
# ==================================================

transform = transforms.Compose([

    transforms.Resize(
        (image_size, image_size)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor()
])

## 04. 전체 데이터 확인

In [5]:
# ==================================================
# 4. 데이터셋 확인
# ==================================================

dataset = datasets.ImageFolder(
    dataset_path,
    transform=transform
)


print("AI가 알고 있는 물건:")
print(dataset.classes)

print()

print("전체 이미지:")
print(len(dataset))

AI가 알고 있는 물건:
['nipper', 'pen', 'wire stripper']

전체 이미지:
271


## 05. CNN 모델 정의

In [6]:
nn.Linear(128, 2)

Linear(in_features=128, out_features=2, bias=True)

In [7]:
# ==================================================
# 5. CNN 모델 정의
# ==================================================

class CNN(nn.Module):

    def __init__(self):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                3,
                16,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),


            nn.Conv2d(
                16,
                32,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )


        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 16 * 16,
                128
            ),

            nn.ReLU(),

            # 0 = 다른 물건
            # 1 = 목표 물건
            nn.Linear(
                128,
                2
            )
        )


    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

## 06. 부품별 이진분류 데이터 만들기

In [8]:
target_class = "nipper"

In [9]:
# ==================================================
# 6. 부품별 이진분류 데이터 만들기
# ==================================================

def make_binary_dataset(
    target_class
):

    image_paths = []

    labels = []


    # 모든 이미지 확인
    for image_path, class_index in dataset.samples:

        # 원래 클래스 이름
        original_class = dataset.classes[
            class_index
        ]


        # 목표 물건이면 1
        if original_class == target_class:

            label = 1


        # 나머지 물건이면 0
        else:

            label = 0


        image_paths.append(
            image_path
        )

        labels.append(
            label
        )


    return image_paths, labels

## 07. 이진분류 Dataset 만들기

In [10]:
# ==================================================
# 7. 이진분류 Dataset
# ==================================================

class BinaryObjectDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        image_paths,
        labels,
        transform=None
    ):

        self.image_paths = image_paths

        self.labels = labels

        self.transform = transform


    def __len__(self):

        return len(
            self.image_paths
        )


    def __getitem__(
        self,
        index
    ):

        # 이미지 경로
        image_path = self.image_paths[
            index
        ]


        # 정답
        label = self.labels[
            index
        ]


        # 이미지 불러오기
        image = Image.open(
            image_path
        ).convert("RGB")


        # 이미지 전처리
        if self.transform:

            image = self.transform(
                image
            )


        # 정답을 Tensor로 변환
        label = torch.tensor(
            label,
            dtype=torch.long
        )


        return image, label

## 08. 부품 하나를 학습하는 함수

셀 8

이 코드가 실제로:

* nipper_model.pth
* pen_model.pth
* wire_stripper_model.pth 

를 만드는 부분이야.

In [11]:
# ==================================================
# 8. 부품 하나 학습하기
# ==================================================

def train_one_object(
    target_class
):

    print()

    print("================================")
    print(
        f"🎯 학습 대상: {target_class}"
    )
    print("================================")


    # --------------------------------------------------
    # 이진분류 데이터 만들기
    # --------------------------------------------------

    image_paths, labels = make_binary_dataset(
        target_class
    )


    # --------------------------------------------------
    # 데이터셋 만들기
    # --------------------------------------------------

    binary_dataset = BinaryObjectDataset(

        image_paths,

        labels,

        transform=transform
    )


    # --------------------------------------------------
    # 학습 / 검증 데이터 분리
    # --------------------------------------------------

    train_size = int(
        len(binary_dataset) * 0.8
    )

    val_size = (
        len(binary_dataset)
        - train_size
    )


    train_dataset, val_dataset = random_split(

        binary_dataset,

        [
            train_size,
            val_size
        ]
    )


    # --------------------------------------------------
    # DataLoader
    # --------------------------------------------------

    train_loader = DataLoader(

        train_dataset,

        batch_size=batch_size,

        shuffle=True
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=batch_size,

        shuffle=False
    )


    # --------------------------------------------------
    # 모델 생성
    # --------------------------------------------------

    model = CNN()


    # --------------------------------------------------
    # 손실 함수
    # --------------------------------------------------

    criterion = nn.CrossEntropyLoss()


    # --------------------------------------------------
    # 옵티마이저
    # --------------------------------------------------

    optimizer = optim.Adam(

        model.parameters(),

        lr=learning_rate
    )


    # ==================================================
    # 학습 시작
    # ==================================================

    for epoch in range(epochs):

        model.train()


        total_loss = 0


        # --------------------------------------------------
        # 학습
        # --------------------------------------------------

        for images, labels in train_loader:

            # 예측
            prediction = model(
                images
            )


            # 손실 계산
            loss = criterion(
                prediction,
                labels
            )


            # 기존 기울기 초기화
            optimizer.zero_grad()


            # 미분
            loss.backward()


            # 가중치 업데이트
            optimizer.step()


            total_loss += loss.item()


        # --------------------------------------------------
        # 검증
        # --------------------------------------------------

        model.eval()


        correct = 0

        total = 0


        with torch.no_grad():

            for images, labels in val_loader:

                prediction = model(
                    images
                )


                _, predicted = torch.max(
                    prediction,
                    dim=1
                )


                total += labels.size(0)


                correct += (
                    predicted == labels
                ).sum().item()


        accuracy = correct / total


        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"Loss: {total_loss:.4f} "
            f"Val Accuracy: "
            f"{accuracy * 100:.1f}%"
        )


    # ==================================================
    # 모델 저장
    # ==================================================

    save_path = os.path.join(

        model_path,

        f"{target_class}_model.pth"
    )


    torch.save(

        {
            "model_state":
                model.state_dict(),

            "target_class":
                target_class,

            "classes": [
                "other",
                target_class
            ]
        },

        save_path
    )


    print()

    print("✅ 학습 완료!")

    print(
        f"✅ 모델 저장: {save_path}"
    )


    return model

## 09. 먼저 nipper 하나만 학습

In [12]:
# ==================================================
# 9. nipper 모델 학습
# ==================================================

nipper_model = train_one_object(
    "nipper"
)


🎯 학습 대상: nipper


/home/user14/Documents/GitHub/DeepLearningProject/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:484: UserWarning: Found GPU0 NVIDIA GeForce GT 1030 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.14.0+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.14.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/user14/Documents/GitHub/DeepLearningProject/.venv/lib/python3.12/site-packages/torch/cuda/__

Epoch 1/20 Loss: 32.4211 Val Accuracy: 81.8%
Epoch 2/20 Loss: 26.8750 Val Accuracy: 83.6%
Epoch 3/20 Loss: 17.8331 Val Accuracy: 90.9%
Epoch 4/20 Loss: 10.8162 Val Accuracy: 92.7%
Epoch 5/20 Loss: 8.7284 Val Accuracy: 98.2%
Epoch 6/20 Loss: 5.4214 Val Accuracy: 98.2%
Epoch 7/20 Loss: 2.7635 Val Accuracy: 98.2%
Epoch 8/20 Loss: 0.9219 Val Accuracy: 98.2%
Epoch 9/20 Loss: 0.3402 Val Accuracy: 98.2%
Epoch 10/20 Loss: 0.3734 Val Accuracy: 98.2%
Epoch 11/20 Loss: 0.1239 Val Accuracy: 98.2%
Epoch 12/20 Loss: 0.0270 Val Accuracy: 98.2%
Epoch 13/20 Loss: 0.0804 Val Accuracy: 98.2%
Epoch 14/20 Loss: 3.3012 Val Accuracy: 98.2%
Epoch 15/20 Loss: 4.3571 Val Accuracy: 98.2%
Epoch 16/20 Loss: 3.4204 Val Accuracy: 98.2%
Epoch 17/20 Loss: 3.4007 Val Accuracy: 98.2%
Epoch 18/20 Loss: 1.7815 Val Accuracy: 98.2%
Epoch 19/20 Loss: 1.4910 Val Accuracy: 98.2%
Epoch 20/20 Loss: 0.0523 Val Accuracy: 98.2%

✅ 학습 완료!
✅ 모델 저장: models/nipper_model.pth


## 10. 나머지 모델 학습


In [13]:
# ==================================================
# 10. 나머지 부품 학습
# ==================================================

pen_model = train_one_object(
    "pen"
)


wire_stripper_model = train_one_object(
    "wire_stripper"
)


🎯 학습 대상: pen
Epoch 1/20 Loss: 23.5383 Val Accuracy: 100.0%
Epoch 2/20 Loss: 3.7683 Val Accuracy: 100.0%
Epoch 3/20 Loss: 15.2630 Val Accuracy: 98.2%
Epoch 4/20 Loss: 3.0393 Val Accuracy: 92.7%
Epoch 5/20 Loss: 1.6389 Val Accuracy: 98.2%
Epoch 6/20 Loss: 2.4138 Val Accuracy: 98.2%
Epoch 7/20 Loss: 0.1239 Val Accuracy: 100.0%
Epoch 8/20 Loss: 0.3626 Val Accuracy: 98.2%
Epoch 9/20 Loss: 0.0304 Val Accuracy: 100.0%
Epoch 10/20 Loss: 0.0382 Val Accuracy: 100.0%
Epoch 11/20 Loss: 0.0218 Val Accuracy: 100.0%
Epoch 12/20 Loss: 0.0968 Val Accuracy: 100.0%
Epoch 13/20 Loss: 0.0418 Val Accuracy: 100.0%
Epoch 14/20 Loss: 0.0031 Val Accuracy: 100.0%
Epoch 15/20 Loss: 0.0037 Val Accuracy: 100.0%
Epoch 16/20 Loss: 0.0012 Val Accuracy: 100.0%
Epoch 17/20 Loss: 0.0027 Val Accuracy: 100.0%
Epoch 18/20 Loss: 0.0013 Val Accuracy: 100.0%
Epoch 19/20 Loss: 0.0012 Val Accuracy: 100.0%
Epoch 20/20 Loss: 0.0019 Val Accuracy: 100.0%

✅ 학습 완료!
✅ 모델 저장: models/pen_model.pth

🎯 학습 대상: wire_stripper
Epoch 1/20 Los